# Izrada aplikacija za generiranje slika (Azure OpenAI)

LLM-ovi nisu samo za generiranje teksta. Također možete generirati slike iz tekstualnih opisa. Slike kao modalitet korisne su u MedTech-u, arhitekturi, turizmu, razvoju igara, marketingu i drugim područjima. U ovoj lekciji radimo s današnjim **GPT Image** modelima na Microsoft Foundryju.

## Ciljevi učenja

Na kraju ove lekcije moći ćete:

- Objasniti što je generiranje slika i gdje je korisno.
- Razumjeti obitelj modela `gpt-image` i kako se razlikuje od naslijeđenih DALL·E modela.
- Izraditi aplikaciju za generiranje slika i uređivati slike.

## Što je generiranje slika?

Modeli za generiranje slika stvaraju slike na temelju tekstualnog upita. Moderni modeli poput `gpt-image` uče odnos između teksta i slika tijekom treniranja, zatim iterativno pretvaraju slučajni šum u sliku koja odgovara vašem upitu.

Dvije poznate obitelji modela za slike su:

- **`gpt-image` (OpenAI)** — trenutna generacija korištena u ovoj lekciji. Podržava generiranje slike iz teksta i uređivanje slika (popunjavanje slike s maskom).
- **Midjourney** — popularan model treće strane s vlastitom uslugom i radnim procesom temeljenim na Discordu.

> Stariji OpenAI modeli slika — **DALL·E 2** i **DALL·E 3** — su naslijeđeni. DALL·E 3 više nije dostupan za nove implementacije, a značajke poput `create_variation` postojale su samo u DALL·E 2. Za nove aplikacije koristite `gpt-image` modele.

Na Microsoft Foundryju, **`gpt-image-2`** je najnoviji i najsposobniji model za slike i preporučeni zadani izbor. `gpt-image-1.5` i `gpt-image-1-mini` su također općenito dostupni.

> **Važno:** `gpt-image` modeli vraćaju generiranu sliku kao **base64** (`b64_json`), a ne kao URL. Vaš kod dekodira base64 niz u bajtove i sprema sliku — nema URL za preuzimanje slike.


## Izrada vaše prve aplikacije za generiranje slika

Pa što je potrebno za izradu aplikacije za generiranje slika? Trebate sljedeće biblioteke:

- **python-dotenv**, preporučuje se koristiti ovu biblioteku za čuvanje vaših tajni u *.env* datoteci odvojeno od koda.
- **openai**, ovu biblioteku ćete koristiti za interakciju s OpenAI API-jem.
- **pillow**, za rad sa slikama u Pythonu.

Ako to već niste učinili, slijedite upute na stranici [Microsoft Learn](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/create-resource?pivots=web-portal&WT.mc_id=academic-105485-koreyst) za stvaranje Microsoft Foundry resursa i modela. Odaberite **gpt-image-2** kao model (najnoviji Azure OpenAI model za slike; DALL·E je zastario).

1. Izradite datoteku *.env* sa sljedećim sadržajem:

    ```text
    AZURE_OPENAI_ENDPOINT=<your endpoint>
    AZURE_OPENAI_API_KEY=<your key>
    AZURE_OPENAI_DEPLOYMENT="gpt-image-2"
    ```

    Potražite ove informacije u Microsoft Foundry portalu za vaš resurs u odjeljku "Deployments".



1. Navedene biblioteke spremite u datoteku pod nazivom *requirements.txt* na sljedeći način:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. Zatim, kreirajte virtualno okruženje i instalirajte biblioteke:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> Za Windows koristite sljedeće naredbe za kreiranje i aktiviranje vašeg virtualnog okruženja:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. Dodajte sljedeći kod u datoteku nazvanu *app.py*:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError

    # import dotenv
    dotenv.load_dotenv()

    # konfigurirajte Azure OpenAI (Microsoft Foundry) klijenta.
    # Modeli za slike zahtijevaju nedavnu verziju API-ja — provjerite Foundry dokumentaciju za verziju koju vaš model treba.
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )

    try:
        # Kreirajte sliku koristeći API za generiranje slika
        generation_response = client.images.generate(
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Unesite tekst vašeg upita ovdje
            size='1024x1024',
            n=1,
            model=os.environ['AZURE_OPENAI_DEPLOYMENT']   # npr. "gpt-image-2"
        )
        # Postavite direktorij za pohranu slike
        image_dir = os.path.join(os.curdir, 'images')

        # Ako direktorij ne postoji, kreirajte ga
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # Inicijalizirajte putanju slike (obratite pažnju da tip datoteke treba biti png)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # gpt-image modeli vraćaju sliku kao base64 (b64_json), a ne URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Prikažite sliku u zadanoj pregledniku slika
        image = Image.open(image_path)
        image.show()

    # uhvati iznimke
    except BadRequestError as err:
        print(err)

    ```

Objasnimo ovaj kod:

- Prvo uvozimo biblioteke koje trebamo, uključujući OpenAI biblioteku, dotenv biblioteku, base64 modul i Pillow biblioteku.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError
    ```

- Zatim učitavamo varijable okoline iz datoteke *.env*.

    ```python
    # uvoz dotenv
    dotenv.load_dotenv()
    ```

- Nakon toga konfiguriramo Azure OpenAI (Microsoft Foundry) klijent.

    ```python
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )
    ```

- Zatim generiramo sliku:

    ```python
    generation_response = client.images.generate(
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Unesite svoj tekst upita ovdje
        size='1024x1024',
        n=1,
        model=os.environ['AZURE_OPENAI_DEPLOYMENT']
    )
    ```

    `gpt-image` modeli vraćaju sliku kao **base64** string u `data[0].b64_json`. Dekodiramo je u bajtove i zapisujemo u datoteku — nema URL-a za preuzimanje.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- Na kraju, otvaramo sliku i koristimo standardni preglednik slika da je prikažemo:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### Više detalja o generiranju slike

Pogledajmo parametre `images.generate`:

- **prompt**, tekstualni prompt koji se koristi za generiranje slike. Ovdje je "Zec na konju, drži lizalicu, na maglenoj livadi gdje rastu narcisi".
- **size**, veličina generirane slike (na primjer `1024x1024`, `1536x1024`, `1024x1536` ili `"auto"`).
- **n**, broj generiranih slika. Ovdje generiramo jednu.
- **model**, ime vašeg modela za slike (na primjer `gpt-image-2`).

> Modeli za slike ne koriste parametar `temperature` — to je kontrola za generiranje teksta. Za raznolikost, pozovite API ponovno; za smanjenje raznolikosti, učinite vaš prompt specifičnijim.

## Dodatne mogućnosti generiranja slike

Vidjeli ste kako generirati sliku s nekoliko linija Pythona. `gpt-image` modeli također mogu **uređivati** postojeću sliku. Pružanjem slike, opcionalne **maske** (koja označava područje za promjenu) i prompta, možete promijeniti dio slike — na primjer, dodati šešir našem zecu.

```python
response = client.images.edit(
  model=os.environ['AZURE_OPENAI_DEPLOYMENT'],
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# uređivanja se također vraćaju kao base64
edited_image = base64.b64decode(response.data[0].b64_json)
```

Osnovna slika sadrži samo zeca; konačna slika ima šešir na zecu.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Napomena**:
Ovaj dokument je preveden korištenjem AI prevoditeljskog servisa [Co-op Translator](https://github.com/Azure/co-op-translator). Iako težimo točnosti, imajte na umu da automatski prijevodi mogu sadržavati greške ili netočnosti. Izvorni dokument na izvornom jeziku treba smatrati autoritativnim izvorom. Za važne informacije preporuča se profesionalni ljudski prijevod. Nismo odgovorni za bilo kakva nesporazumevanja ili pogrešne interpretacije koje proizlaze iz korištenja ovog prijevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
